# chemdatacheck quickstart
## pytest for molecular datasets

In 10 minutes you will:

1. Build a deliberately dirty dataset.
2. Run `chemdatacheck` from the command line and read the report.
3. Do the same audit from Python and inspect a single finding.
4. Export machine-readable output and gate on exit codes, like pytest.

> **Prerequisites:** ChemDataCheck requires Python 3.10 or newer and is tested with
> Python 3.10–3.12. Follow the environment setup in the
> [README](../README.md#install), using
> `python -m pip install ".[pretty,notebooks]"` as the installation command.
> Start Jupyter from the same activated environment.
>
> This notebook runs offline: all data is generated in the first code cell.

## 1. Make a deliberately dirty dataset

Each planted fault maps to one chemdatacheck finding. In real life, these all look innocent in a spreadsheet.

In [1]:
import pandas as pd

rows = [
    # id,      smiles,            activity, split    # planted fault
    ("d01",    "CCO",             1.2,      "train"),  # ok (ethanol)
    ("d02",    "CCO",             1.2,      "train"),  # exact duplicate of d01
    ("d03",    "c1ccccc1",        2.0,      "train"),  # ok (benzene)
    ("d04",    "C[C@H](Cl)Br",    3.0,      "train"),  # one stereoisomer ...
    ("d05",    "C[C@@H](Cl)Br",   3.5,      "test"),   # ... the other one, across the split!
    ("d06",    "CC(Cl)Br",        3.1,      "test"),   # same connectivity, stereo unspecified
    ("d07",    "CCO.[Na+]",        1.2,      "test"),   # ethanol again, as a sodium salt
    ("d08",    "not_a_smiles!!",  0.0,      "train"),  # garbage string
    ("d09",    "C(C)(C)(C)(C)C",  0.5,      "train"),  # carbon with 5 bonds
    ("d10",    "[C-2]",            0.1,      "train"),  # impossible charge on carbon
    ("d11",    "CC(=O)C",         4.0,      "train"),  # keto form ...
    ("d12",    "CC(O)=C",         5.5,      "train"),  # ... vs enol form (tautomers!)
    ("d13",    "[13C]C",          1.0,      "train"),  # isotopic label
    ("d14",    "c1ccccc1O",       2.5,      "test"),   # phenol, measured twice ...
    ("d15",    "c1ccccc1O",       7.9,      "test"),   # ... with wildly different values!
    ("d16",    "CCCCCCCCO",       1.1,      "train"),  # octanol ...
    ("d17",    "CCCCCCCCCO",      1.3,      "test"),   # ... vs nonanol, across the split
    ("d18",    "CC(=O)O",         9.9,      "train"),  # ok (acetic acid)
    ("d19",    "[U]C",            0.2,      "train"),  # uranium?! (rare element)
    ("d20",    "C1OC1CCl",        1.4,      "test"),   # epoxide (rare functional group)
]
df = pd.DataFrame(rows, columns=["id", "smiles", "activity", "split"])
df.to_csv("dirty.csv", index=False)
print(df.shape)
df.head(8)

(20, 4)


,id,smiles,activity,split
0,d01,CCO,1.2,train
1,d02,CCO,1.2,train
2,d03,c1ccccc1,2.0,train
3,d04,C[C@H](Cl)Br,3.0,train
4,d05,C[C@@H](Cl)Br,3.5,test
5,d06,CC(Cl)Br,3.1,test
6,d07,CCO.[Na+],1.2,test
7,d08,not_a_smiles!!,0.0,train


## 2. Run the CLI

One command. This is the whole pitch:

In [ ]:
import sys
!{sys.executable} -m chemdatacheck dirty.csv --label-col activity --split-col split


✓ 18 valid molecules / ✕ 2 invalid (n=20)
✕  near-duplicates across splits (Tc≥0.95): 5 (25.0%)  
    e.g. {"train_row": "d04", "test_row": "d05", "tanimoto": 1.0}
    e.g. {"train_row": "d04", "test_row": "d06", "tanimoto": 1.0}
    e.g. {"train_row": "d16", "test_row": "d17", "tanimoto": 1.0}
    → Near-duplicates must live in one split only — move the test copy to train 
or drop it; otherwise metrics measure memorization.
    (3 cross-split pairs above threshold.)
✕  invalid SMILES: 2 (10.0%)  
    e.g. {"row": "d08", "smiles": "not_a_smiles!!", "error": "RDKit could not 
parse SMILES"}
    e.g. {"row": "d09", "smiles": "C(C)(C)(C)(C)C", "error": 
"AtomValenceException: Explicit valence for atom # 0 C, 5, is greater than 
permitted "}
    → Drop or fix these rows; they cannot be parsed and will silently shrink 
training data.
✕  2D-identity leakage (2 test molecules seen in train): 2 (10.0%)  
    e.g. {"train_rows": ["d04"], "test_row": "d05", "connectivity": "CC(Cl)Br"}
    e.g. 

## 3. Reading the report

Every finding has the same anatomy:

- **severity**: `error` (must fix), `warning` (probably fix), `info` (know about it)
- **check_id**: stable machine-readable name, e.g. `identity_leakage`
- **count / rate**: how many rows are affected
- **affected_rows**: *which* rows (your IDs, not ours)
- **examples**: the evidence (molecule pairs, similarity scores, scaffolds)
- **recommendation**: what to do about it

Let's do the same audit from Python and look at one finding up close:

In [3]:
from chemdatacheck.report import audit, render_terminal

report = audit(["dirty.csv"], label_col="activity", split_col="split")
print(f"score: {report.score}/100   valid: {report.n_valid}/{report.n_total}")
print(f"{'severity':<8} {'count':>5}  check_id")
for f in report.findings:
    print(f"{f.severity.value:<8} {f.count:>5}  {f.check_id}")

score: 39/100   valid: 18/20
severity count  check_id
error        5  cross_split_near_duplicates
error        2  invalid_smiles
error        2  identity_leakage
error        2  conflicting_measurements
error        1  valence_error
warning      9  near_duplicates
warning      6  scaffold_overlap
warning      4  exact_duplicates
warning      4  canonical_duplicates
warning      4  analog_leakage
warning      4  suspiciously_easy_split
warning      3  stereochemical_collisions
warning      3  salt_duplicates
warning      2  radical_flag
warning      2  unspecified_stereo
warning      2  duplicated_measurements
warning      1  impossible_charge
warning      1  disconnected_components
info         5  tautomer_ambiguity
info         5  applicability_gaps
info         2  tautomer_duplicates
info         2  rare_elements
info         1  isotope_flag
info         1  unusual_ring_systems


In [4]:
import json

leak = next(f for f in report.findings if f.check_id == "identity_leakage")
print("TITLE:", leak.title)
print("RECOMMENDATION:", leak.recommendation)
print("EVIDENCE:")
print(json.dumps(leak.examples[0], indent=2))

TITLE: 2D-identity leakage (2 test molecules seen in train)
RECOMMENDATION: Remove leaked test molecules or move them to train; identical structures must never span splits.
EVIDENCE:
{
  "train_rows": [
    "d04"
  ],
  "test_row": "d05",
  "connectivity": "CC(Cl)Br"
}


That is the core chemdatacheck loop: **a warning is not just a count**. It points at rows, shows the offending pair, and tells you the fix.

## 4. Machine-readable output and CI gates

The terminal report is for humans. For pipelines, emit JSON and gate on the exit code, exactly like pytest:

In [5]:
from chemdatacheck.report import render_json
from chemdatacheck.cli import main as chemdatacheck_main

# JSON report: mine it, attach it to a PR, diff it between dataset versions
info = json.loads(render_json(report))
print("summary:", info["summary"])
print("worst deduction:", info["score_breakdown"][0])

# Exit codes: 0 clean, 1 warnings, 2 errors. CI fails the build on errors:
code = chemdatacheck_main(["dirty.csv", "--label-col", "activity", "--split-col", "split",
                       "--format", "json", "--output", "/tmp/chemdatacheck_report.json",
                       "--fail-on", "error", "-q"])
print("exit code:", code, "(use --format junit for CI dashboards)")

summary: {'n_total': 20, 'n_valid': 18, 'n_invalid': 2, 'score': 39}
worst deduction: {'check_id': 'conflicting_measurements', 'title': 'conflicting measurements (same structure, different labels)', 'severity': 'error', 'count': 2, 'weight': 8.0, 'raw_deduction': 5.93, 'family': None, 'deduction': 5.93, 'overlap_adjusted': False}
exit code: 2 (use --format junit for CI dashboards)


## Sanity checks (these double as the notebook's tests)

In [6]:
ids = {f.check_id for f in report.findings}
assert report.n_total == 20
assert {"invalid_smiles", "identity_leakage", "conflicting_measurements",
        "salt_duplicates", "tautomer_duplicates"} <= ids
assert report.score < 100
print("all expected findings fired. Total:", len(ids))

all expected findings fired. Total: 24


## Where next

- **Notebook 02** — *Leakage deep-dive*: identity vs analog vs scaffold leakage, and how to fix each.
- **Notebook 03** — *Curation case study*: take this dirty file to a 90+ score, step by step, and export the HTML report.